In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from dataset import Dataset, process_test_dataset
from model import ElasticityModel
from model_utils import expand_elasticity_matrix_to_original_skus, calculate_metrics, print_metrics_summary

In [ ]:
sellout_df = pd.read_csv('../data/Sellout_Train.csv')
ihs_df = pd.read_csv('../data/9.18.25_IHS Data_trimmed_v0.1.csv')

In [ ]:
%%time
data = Dataset(sellout_df, ihs_df, 18)

In [ ]:
data_dict = data.process_data()

In [ ]:
train_data_full = {
    'log_prices': tf.constant(data_dict['log_prices'], dtype=tf.float32),
    'features': tf.constant(data_dict['features'], dtype=tf.float32),
    'log_volumes': tf.constant(data_dict['log_volumes'], dtype=tf.float32)
}

train_data = {
    'log_prices': tf.constant(data_dict['log_prices'][:30], dtype=tf.float32),
    'features': tf.constant(data_dict['features'][:30], dtype=tf.float32),
    'log_volumes': tf.constant(data_dict['log_volumes'][:30], dtype=tf.float32)
}

val_data = {
    'log_prices': tf.constant(data_dict['log_prices'][30:], dtype=tf.float32),
    'features': tf.constant(data_dict['features'][30:], dtype=tf.float32),
    'log_volumes': tf.constant(data_dict['log_volumes'][30:], dtype=tf.float32)
}

In [ ]:
 model = ElasticityModel(
        n_skus=len(data_dict['effective_skus']), 
        n_features=data_dict['n_features'], 
        learning_rate=0.001,
        reg_lambda=0.1
    )
    
history = model.train(train_data=train_data_full, epochs=50000)

In [ ]:
# # Get final metrics
# final_train_metrics = calculate_metrics(model, train_data, "train")
# final_val_metrics = calculate_metrics(model, val_data, "validation")

# # Print detailed summary
# print_metrics_summary(final_train_metrics, "train")
# print_metrics_summary(final_val_metrics, "validation")

# # Access specific metrics
# train_r2 = final_train_metrics['train_r2_overall']
# val_mape = final_val_metrics['validation_mape_overall']

# print(f"\nFinal Model Performance:")
# print(f"Training R²: {train_r2:.4f}")
# print(f"Validation MAPE: {val_mape:.2f}%")

In [ ]:
effective_elasticity_matrix = model.get_elasticity_matrix()

In [ ]:
original_elasticity_matrix = expand_elasticity_matrix_to_original_skus(
        effective_elasticity_matrix, data
    )
original_elasticity_matrix

In [ ]:
print("Number of SKUs not adhering to the own price elasticity condition:",
    (original_elasticity_matrix[original_elasticity_matrix['elasticity_type']=='own_price']['elasticity']>0).sum()
)

print("Number of SKUs not adhering to the cross price elasticity condition:",
    (original_elasticity_matrix[original_elasticity_matrix['elasticity_type']=='cross_price']['elasticity']<0).sum()
     )

In [ ]:
manufacturer_df = data.sellout_df.copy()
manufacturer_df = manufacturer_df[['sku','manufacturer']]
manufacturer_df

In [ ]:
def add_manufacturer_column(target_df, sku_manufacturer_df, target_sku_col='target_sku', sku_col='sku', manufacturer_col='manufacturer'):

    result_df = target_df.copy()
    sku_to_manufacturer = dict(zip(sku_manufacturer_df[sku_col], sku_manufacturer_df[manufacturer_col]))
    
    result_df[manufacturer_col] = result_df[target_sku_col].map(sku_to_manufacturer)
    unmapped_count = result_df[manufacturer_col].isnull().sum()
    if unmapped_count > 0:
        print(f"Warning: {unmapped_count} SKUs in target dataframe could not be mapped to manufacturers")
        unmapped_skus = result_df[result_df[manufacturer_col].isnull()][target_sku_col].unique()
        print(f"Unmapped SKUs: {unmapped_skus[:10]}...")  # Show first 10 unmapped SKUs
    
    return result_df

final_elasticity_matrix = add_manufacturer_column(original_elasticity_matrix, manufacturer_df, target_sku_col='target_sku', sku_col='sku', manufacturer_col='manufacturer')

In [ ]:
# final_elasticity_matrix
# final_elasticity_matrix.to_csv('../results/elasticity_matrix_v0.1.csv',index=False)